# 05b — Tracker Hyperparameter Tuning: ByteTrack

Attempts a full hyperparameter grid search for ByteTrack, picking up architecture
decisions from `configs/kartector_bytetrack_arch.json` (written by notebook 04b).

> **See the deprecation notice below before running.**

## ⚠️ Deprecated — Not Used in Final Project

Hyperparameter tuning was attempted here and in notebook 05, but the swept
configurations performed **worse** at track association than the architecture
baselines from notebooks 04/04b.

**Decision:** Tracker hyperparameter tuning has been abandoned.
All downstream notebooks (06, 07, 08) use the baseline arch configs from
notebooks 04/04b directly.

These notebooks are retained for reference only.

## 0 — Configuration

In [ ]:
import json as _json
from pathlib import Path
import json, itertools, random
import cv2, numpy as np

REPO_ROOT    = Path('/home1/hendersonj6179@cgu.edu')

# ── Load architecture decisions from notebook 04b ─────────────────────────
_ARCH_CFG = REPO_ROOT / 'configs/kartector_bytetrack_arch.json'
if _ARCH_CFG.exists():
    _arch = _json.loads(_ARCH_CFG.read_text())
    print(f'Loaded arch config from {_ARCH_CFG}:')
    print(_json.dumps(_arch, indent=2))
    CONF            = _arch.get('conf', 0.20)
    POST_PROCESS    = _arch.get('post_process', False)
    MAX_GAP         = _arch.get('max_gap', 45)
    MAX_DIST_PCT    = _arch.get('max_dist_pct', 35.0)
    _TRACKER_YAML   = REPO_ROOT / _arch.get('tracker_yaml', 'configs/kartector_bytetrack_best.yaml')
else:
    print(f'WARNING: {_ARCH_CFG} not found — run 04b_tracker_architecture_bytetrack.ipynb first.')
    print('Falling back to defaults.')
    CONF = 0.20; POST_PROCESS = False; MAX_GAP = 45; MAX_DIST_PCT = 35.0
    _TRACKER_YAML = REPO_ROOT / 'configs/kartector_bytetrack_best.yaml'

print(f'  conf={CONF}  post_process={POST_PROCESS}  max_gap={MAX_GAP}  max_dist_pct={MAX_DIST_PCT}')
MOT_DIR      = REPO_ROOT / 'data/mot_dataset'
YOLO_RUNS    = (REPO_ROOT / 'runs/yolo26').resolve()
TRACKER_RUNS = (REPO_ROOT / 'runs/trackers_bytetrack').resolve()
TRACKER_RUNS.mkdir(parents=True, exist_ok=True)
CHUNKS_DIR   = REPO_ROOT / 'data/processed/labelstudiochunks'
# CONF already loaded from arch config above
IOU = 0.7
# 05b tunes ByteTrack only — compare trackers in 05c_evaluate_compare
TRACKERS = ['bytetrack.yaml']
HP_SEARCH_SUBSET = 0.05
# ── Hyperparameter grid ────────────────────────────────────────────────────
# det_conf is fixed from arch config (set in Section 0 above).
# match_thresh and track_buffer were roughly identified in 04b; here we do a
# finer sweep together with the secondary ByteTrack knobs.
# Post-process settings (MAX_GAP, MAX_DIST_PCT) come from arch config.
# ── Hyperparameter grid ────────────────────────────────────────────────────
# track_buffer and match_thresh are included here for a full fine-tune sweep.
# 04/04b only swept on a single video so wider search is warranted.
# Here we fine-tune the ByteTrack secondary knobs with wider ranges.
HP_GRID = {
    # ── Re-entry / buffering ──────────────────────────────────────────────
    # track_buffer: frames to keep a lost track alive
    'track_buffer':      [10, 20, 30, 45, 60, 90],
    # match_thresh: IoU association threshold
    'match_thresh':      [0.25, 0.50, 0.75, 0.99],
    # ── Detection score thresholds ────────────────────────────────────────
    # track_high_thresh: min score for first-stage association
    'track_high_thresh': [0.10, 0.20, 0.40, 0.60],
    # track_low_thresh: min score for second-stage (recovery) association
    #   must be < track_high_thresh; low values help recover occluded objects
    'track_low_thresh':  [0.01, 0.10,0.20, 0.30, 0.40],
    # new_track_thresh: min score to initialise a brand-new track
    'new_track_thresh':  [0.15, 0.20, 0.25, 0.35, 0.45, 0.55, 0.65],
}
HP_MAX_TRIALS = 100
# HP_SEARCH_METRIC: ACE (count accuracy) | MOTA | IDF1
# BIAS_PENALTY: weight on |mean per-class CE| to penalise systematic bias.
# Composite score (ACE mode) = ACE + BIAS_PENALTY * abs(mean_class_CE)
HP_SEARCH_METRIC = "ACE"
BIAS_PENALTY     = 1.0   # set to 0 to disable bias penalisation
with open(REPO_ROOT / 'data/splits/splits.json') as f:
    splits = json.load(f)
CLASSES = splits['classes']
FINAL_WEIGHTS = YOLO_RUNS / 'final' / 'weights' / 'best.pt'
print('Config OK')
print(f'  YOLO weights: {FINAL_WEIGHTS}')

## 1 — Install boxmot

Only needed if not already installed. boxmot is used for the RT-DETR calibration
section at the bottom of this notebook.

In [ ]:
import subprocess, sys
try:
    import boxmot; print('boxmot ready')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'boxmot', '-q'], check=True)
    import boxmot; print('boxmot installed')

## 2 — Helpers

Post-processing and metric utilities. These are imported from `helpers.py`.

In [ ]:
from helpers import (
    _iou, _cx, _cy,
    _cross_class_nms,
    _merge_track_rows,
    _count_tracks_per_class,
)

## 3 — Hyperparameter Search

In [ ]:
import sys, warnings as _warnings
if not FINAL_WEIGHTS.exists():
    raise FileNotFoundError(f'YOLO weights not found: {FINAL_WEIGHTS}\nRun 02_train_yolo.ipynb first.')

test_chunk_names = splits['mot_test_chunks']
all_test_videos = sorted([CHUNKS_DIR / f for f in test_chunk_names if (CHUNKS_DIR / f).exists()])
print(f'Total test videos: {len(all_test_videos)}')
random.seed(42)
n_sub = max(1, int(len(all_test_videos) * HP_SEARCH_SUBSET))
search_videos = random.sample(all_test_videos, n_sub)
print(f'HP search subset: {len(search_videos)}/{len(all_test_videos)}')

tg = {k: HP_GRID[k] for k in _TRACKER_KEYS if k in HP_GRID}
pg = {k: HP_GRID[k] for k in _PP_KEYS if k in HP_GRID}
all_combos = [dict(**dict(zip(tg, tc)), **dict(zip(pg, pc)))
              for tc, pc in itertools.product(itertools.product(*tg.values()),
                                               itertools.product(*pg.values()))]
random.shuffle(all_combos)
combos = all_combos[:HP_MAX_TRIALS]
print(f'HP trials: {len(combos)}/{len(all_combos)} total')

# ── Results DataFrame ─────────────────────────────────────────────────────
# One row per trial: tracker + all HP params + all metrics.
# Best config = sort df_hp by chosen metric and take first row.
hp_rows = []

import yaml as _ty_b

for tracker in TRACKERS:
    tname = tracker_name(tracker)
    print(f'\n{"="*60}\nHP search — {tname}  ({len(search_videos)} videos)  '
          f'[optimising {HP_SEARCH_METRIC}]\n{"="*60}', flush=True)

    for trial_idx, combo in enumerate(combos):
        t_hp = {k: v for k, v in combo.items() if k in _TRACKER_KEYS}
        p_hp = {k: v for k, v in combo.items() if k in _PP_KEYS}

        # Inject fixed track_buffer & match_thresh from arch config yaml
        _bt_hp = dict(t_hp)
        if _arch and 'tracker_yaml' in _arch:
            _arch_yaml_b = REPO_ROOT / _arch['tracker_yaml']
            if _arch_yaml_b.exists():
                _arch_vals_b = _ty_b.safe_load(_arch_yaml_b.read_text())
                _bt_hp.setdefault('track_buffer', _arch_vals_b.get('track_buffer', 30))
                _bt_hp.setdefault('match_thresh', _arch_vals_b.get('match_thresh', 0.60))
        t_hp = _bt_hp

        # Post-process params come from arch config when POST_PROCESS is on
        if POST_PROCESS:
            p_hp = {'pp_merge_time': MAX_GAP, 'pp_merge_dist': MAX_DIST_PCT,
                    'pp_merge_interp': True, 'pp_cc_iou': None}

        pred_dir = TRACKER_RUNS / tname / 'hp_search' / f'trial_{trial_idx:03d}'
        print(f'  → [{trial_idx+1:3d}/{len(combos)}] {combo}', flush=True)
        try:
            run_tracker(FINAL_WEIGHTS, search_videos, pred_dir, tracker,
                        hp_overrides=t_hp, conf=CONF, pp_overrides=p_hp)
            m = compute_mot_metrics(MOT_DIR / 'test', pred_dir)

            # Count metrics
            df_cnt_trial = compute_count_metrics(MOT_DIR / 'test', pred_dir, CLASSES)
            per_cls_t = df_cnt_trial[df_cnt_trial['class'] != 'MACRO_AVG']
            with _warnings.catch_warnings():
                _warnings.simplefilter('ignore', RuntimeWarning)
                ace_trial  = float(per_cls_t['ACE'].mean()) if not per_cls_t.empty else None
                bias_trial = float(per_cls_t['CE'].mean())  if not per_cls_t.empty else None
            composite_trial = ((ace_trial + BIAS_PENALTY * abs(bias_trial))
                               if (ace_trial is not None and bias_trial is not None) else None)

            row = {
                'tracker':   tname,
                'trial':     trial_idx,
                **combo,
                'MOTA':      m.get('MOTA'),
                'IDF1':      m.get('IDF1'),
                'ID_Sw':     m.get('ID_Sw'),
                'ACE':       ace_trial,
                'CE_bias':   bias_trial,
                'composite': composite_trial,
            }
            hp_rows.append(row)

            def _fmt(v, fmt='.4f'): return format(v, fmt) if v is not None else 'None'
            print(f"      MOTA={_fmt(row['MOTA'])} IDF1={_fmt(row['IDF1'])} "
                  f"ID_Sw={row['ID_Sw']} ACE={_fmt(row['ACE'])} "
                  f"CE_bias={_fmt(row['CE_bias'])} composite={_fmt(row['composite'])}",
                  flush=True)

        except Exception as _trial_err:
            print(f'  [ERROR trial {trial_idx}] {_trial_err}', flush=True)
            continue

# ── Build results DataFrame ───────────────────────────────────────────────
df_hp = pd.DataFrame(hp_rows)
df_hp.to_csv(TRACKER_RUNS / 'hp_search_log.csv', index=False)
print(f'\nSaved {len(df_hp)} trial results → {TRACKER_RUNS}/hp_search_log.csv')

# ── Pick best config per tracker by HP_SEARCH_METRIC ─────────────────────
_METRIC_ASCENDING = {'MOTA': False, 'IDF1': False, 'ID_Sw': True,
                     'ACE': True, 'composite': True, 'CE_bias': True}
_sort_col  = HP_SEARCH_METRIC
_ascending = _METRIC_ASCENDING.get(_sort_col, False)

best_configs = {}   # tracker yaml -> best combo dict (HP params only)
best_metrics = {}   # tracker yaml -> best metrics dict

print(f'\nBest configs (sorted by {_sort_col}, ascending={_ascending}):')
for tracker in TRACKERS:
    tname = tracker_name(tracker)
    sub = df_hp[df_hp['tracker'] == tname].dropna(subset=[_sort_col])
    if sub.empty:
        print(f'  {tname}: no valid trials')
        continue
    best_row = sub.sort_values(_sort_col, ascending=_ascending).iloc[0]
    best_configs[tracker] = {k: best_row[k].item() if hasattr(best_row[k], 'item') else best_row[k] for k in combo if k in best_row.index}
    best_metrics[tracker] = {k: best_row[k]
                              for k in ['MOTA','IDF1','ID_Sw','ACE','CE_bias','composite']}
    print(f'  {tname}:')
    print(f'    HP     : { {k:v for k,v in best_configs[tracker].items() if k in _TRACKER_KEYS} }')
    print(f'    Metrics: {best_metrics[tracker]}')

print('\ndf_hp preview:')
display(df_hp.sort_values(['tracker', _sort_col], ascending=[True, _ascending]).head(20))


## 4 — Final Evaluation

In [ ]:
final_results = {}
eval_videos = search_videos  # swap to all_test_videos for full eval
for tracker in TRACKERS:
    tname   = tracker_name(tracker)
    best_hp = best_configs.get(tracker, {})
    t_hp    = {k: v for k, v in best_hp.items() if k in _TRACKER_KEYS}
    p_hp    = {k: v for k, v in best_hp.items() if k in _PP_KEYS}
    pred_dir = TRACKER_RUNS / tname / 'final_eval'
    print(f'Running {tname} ({len(eval_videos)} videos)...')
    # Inject arch fixed knobs
    import yaml as _ty3
    _fe_hp = dict(t_hp)
    if _arch and 'tracker_yaml' in _arch:
        _av3 = _ty3.safe_load((REPO_ROOT/_arch['tracker_yaml']).read_text())
        _fe_hp.setdefault('track_buffer', _av3.get('track_buffer', 30))
        _fe_hp.setdefault('match_thresh', _av3.get('match_thresh', 0.60))
    pp_hp_fe = {}
    if POST_PROCESS:
        pp_hp_fe = {'pp_merge_time': MAX_GAP, 'pp_merge_dist': MAX_DIST_PCT,
                    'pp_merge_interp': True, 'pp_cc_iou': None}
    run_tracker(FINAL_WEIGHTS, eval_videos, pred_dir, tracker,
                hp_overrides=_fe_hp, conf=CONF, pp_overrides=pp_hp_fe)
    m = compute_mot_metrics(MOT_DIR / 'test', pred_dir)
    final_results[tname] = {'metrics': m, 'hp': best_hp, 'pred_dir': str(pred_dir)}
    print(f"  -> MOTA={m['MOTA']}  IDF1={m['IDF1']}  ID_Sw={m['ID_Sw']}")

df_final = pd.DataFrame([{'tracker': k, **v['metrics']} for k, v in final_results.items()])
print(df_final.round(4).to_string(index=False))
df_final.to_csv(TRACKER_RUNS / 'test_metrics.csv', index=False)
with open(TRACKER_RUNS / 'test_metrics.json', 'w') as f:
    json.dump({k: v['metrics'] for k, v in final_results.items()}, f, indent=2)
with open(TRACKER_RUNS / 'best_configs.json', 'w') as f:
    json.dump({k: v['hp'] for k, v in final_results.items()}, f, indent=2)

## 4b — Per-Class Count Metrics

Same metrics as notebook 05 (CE, ACE, CA) for ByteTrack.

In [ ]:
# _count_tracks_per_class already imported above from helpers

## 5 — Metric Visualizations

In [ ]:
tnames = [tracker_name(t) for t in TRACKERS]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#4C72B0', '#DD8452']

for ax, metric in zip(axes, ['MOTA', 'IDF1', 'ID_Sw']):
    vals = [final_results[n]['metrics'].get(metric) or 0 for n in tnames]
    bars = ax.bar(tnames, vals, color=colors[:len(tnames)])
    ax.set_title(metric)
    ax.set_ylim(0, max(vals) * 1.3 if max(vals) > 0 else 1)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.3f}' if metric != 'ID_Sw' else str(int(val)),
                ha='center', va='bottom', fontsize=10)

plt.suptitle('RT-DETR Tracker Comparison (best HP)', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(TRACKER_RUNS / 'metric_barplot.png', dpi=150, bbox_inches='tight')
plt.show()

import seaborn as sns
df_hp_log = df_hp
if 'track_high_thresh' in HP_GRID and 'match_thresh' in HP_GRID and not df_hp_log.empty:
    fig, axes = plt.subplots(1, len(TRACKERS), figsize=(7 * len(TRACKERS), 5))
    if len(TRACKERS) == 1: axes = [axes]
    for ax, tracker in zip(axes, TRACKERS):
        name = tracker_name(tracker)
        sub  = df_hp_log[df_hp_log['tracker'] == name].copy()
        if sub.empty or sub['MOTA'].isna().all():
            ax.set_title(f'{name} — no data'); continue
        pivot = sub.pivot_table(index='track_high_thresh', columns='match_thresh',
                                 values='MOTA', aggfunc='mean')
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax,
                    linewidths=0.5, cbar_kws={'label': 'MOTA'})
        ax.set_title(f'{name} — MOTA (track_high_thresh x match_thresh)')
    plt.tight_layout()
    plt.savefig(TRACKER_RUNS / 'hp_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {TRACKER_RUNS}/hp_heatmap.png')

## 5b — Per-Class Count Metrics Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('RT-DETR Per-Class Count Metrics — Test Set', fontweight='bold')
bar_colors = ['#4C72B0', '#DD8452']

for ax, metric, ylabel in zip(axes,
                                ['CE', 'ACE', 'CA'],
                                ['Count Error (CE)', 'Abs Count Error (ACE)', 'Count Accuracy (CA)']):
    class_labels = CLASSES
    x     = np.arange(len(class_labels))
    width = 0.8 / len(TRACKERS)
    for ti, tracker in enumerate(TRACKERS):
        tname = tracker_name(tracker)
        df    = count_results[tname]
        vals  = [df.loc[df['class'] == c, metric].values[0]
                 if c in df['class'].values else 0 for c in class_labels]
        offset = (ti - len(TRACKERS) / 2 + 0.5) * width
        ax.bar(x + offset, vals, width, label=tname,
               color=bar_colors[ti % len(bar_colors)], alpha=0.85)
    if metric == 'CE':
        ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_xticks(x); ax.set_xticklabels(class_labels, rotation=30, ha='right')
    ax.set_ylabel(ylabel); ax.set_title(ylabel)
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
out_path = TRACKER_RUNS / 'count_metrics.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 6 — Annotated Frame Snapshots (YOLO)

12 evenly-spaced frames from the first eval video with YOLO+ByteTrack tracks overlaid.

In [ ]:
import math

demo_video = eval_videos[0] if eval_videos else None

def annotated_grid_yolo(video_path, weights_path, best_hp, n_frames=12):
    hp = dict(best_hp)
    t_hp = {k: v for k, v in hp.items() if k in _TRACKER_KEYS}
    tmp = _make_tracker_yaml_bt(t_hp)
    model = _YOLO_BT(str(weights_path))
    cap   = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    results_list = list(model.track(
        source=str(video_path), conf=CONF, iou=IOU,
        tracker=str(tmp), persist=False, verbose=False, stream=True))
    if tmp.exists(): tmp.unlink()
    indices = [int(i * (total - 1) / (n_frames - 1)) for i in range(n_frames)]
    out_frames = []
    cap = cv2.VideoCapture(str(video_path))
    for fi in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if not ret: continue
        r = results_list[min(fi, len(results_list) - 1)]
        boxes, tids, cids, confs = [], [], [], []
        if r.boxes is not None and r.boxes.id is not None:
            for box in r.boxes:
                if box.id is None: continue
                boxes.append(box.xyxy[0].tolist())
                tids.append(int(box.id.item()))
                cids.append(int(box.cls.item()))
                confs.append(float(box.conf.item()))
        out_frames.append(draw_tracks(frame, boxes, tids, cids, CLASSES, confs))
    cap.release()
    return out_frames

if demo_video:
    for tracker in TRACKERS:
        tname   = tracker_name(tracker)
        best_hp = best_configs.get(tracker, {})
        frames  = annotated_grid_yolo(demo_video, FINAL_WEIGHTS, best_hp, n_frames=12)
        cols  = 4; rows_n = math.ceil(len(frames) / cols)
        fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 4, rows_n * 3))
        axes = axes.flatten()
        for ax, fr in zip(axes, frames):
            ax.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); ax.axis('off')
        for ax in axes[len(frames):]: ax.axis('off')
        plt.suptitle(f'YOLO+{tname} — annotated frames  ({Path(demo_video).name})', fontsize=12)
        plt.tight_layout()
        out_path = TRACKER_RUNS / f'annotated_yolo_{tname}.png'
        plt.savefig(out_path, dpi=120, bbox_inches='tight'); plt.show()
        print(f'Saved: {out_path}')
else:
    print('No eval video found — skipping frame snapshots.')

## 7 — Trajectory Plot (YOLO)

Centroid paths for the best-ACE ByteTrack config (YOLO) on the demo video.

In [ ]:
from helpers import _load_gt_trajectories, _draw_trajectory_ax

## Save YOLO/ByteTrack Final Config

In [ ]:
# ── Save YOLO final config (ByteTrack) ───────────────────────────────────────
import json as _jsave, yaml as _ysave

_bt_best    = best_configs.get("bytetrack.yaml", {})
_bt_hp      = {k: v for k, v in _bt_best.items() if k in _TRACKER_KEYS}

# Load fixed knobs from arch yaml and merge
_arch_yaml_path = REPO_ROOT / _arch['tracker_yaml']
_arch_yaml_vals = _ysave.safe_load(_arch_yaml_path.read_text()) if _arch_yaml_path.exists() else {}
_bt_hp.setdefault('track_buffer', _arch_yaml_vals.get('track_buffer', 30))
_bt_hp.setdefault('match_thresh', _arch_yaml_vals.get('match_thresh', 0.60))

# Write the final tuned tracker YAML
_final_yaml = REPO_ROOT / 'configs' / 'kartector_bytetrack_yolo_final.yaml'
_final_yaml_dict = {
    'tracker_type':      'bytetrack',
    'track_high_thresh': _bt_hp.get('track_high_thresh', 0.25),
    'track_low_thresh':  _bt_hp.get('track_low_thresh', 0.10),
    'new_track_thresh':  _bt_hp.get('new_track_thresh', 0.30),
    'track_buffer':      int(_bt_hp.get('track_buffer', 30)),
    'match_thresh':      float(_bt_hp.get('match_thresh', 0.60)),
    'fuse_score':        True,
}
_final_yaml.write_text(_ysave.dump(_final_yaml_dict))

# Write the full workflow JSON
_yolo_cfg = {
    'backbone':      'yolo',
    'weights':       str((YOLO_RUNS / 'final' / 'weights' / 'best.pt').relative_to(REPO_ROOT)),
    'tracker':       'bytetrack',
    'tracker_yaml':  str(_final_yaml.relative_to(REPO_ROOT)),
    'conf':          float(CONF),
    'iou':           float(IOU),
    'post_process':  bool(POST_PROCESS),
    'max_gap':       int(MAX_GAP),
    'max_dist_pct':  float(MAX_DIST_PCT),
    'best_hp':       _bt_hp,
}
_out = REPO_ROOT / 'configs' / 'kartector_bytetrack_yolo_final.json'
_out.write_text(_jsave.dumps(_yolo_cfg, indent=2))
print(f'YOLO/ByteTrack config saved: {_out}')
print(_jsave.dumps(_yolo_cfg, indent=2))


## RT-DETR Score Threshold Calibration

Same as notebook 05 — motion knobs are frozen to the best ByteTrack values;
only confidence/score thresholds are swept for RT-DETR.

In [ ]:
from ultralytics import RTDETR as _RTDETR_B
import tempfile, yaml as _yaml_b

RTDETR_RUNS_B        = (REPO_ROOT / 'runs/rtdetr').resolve()
FINAL_RTDETR_WEIGHTS = RTDETR_RUNS_B / 'final' / 'weights' / 'best.pt'
YOLO_RUNS            = (REPO_ROOT / 'runs/yolo26').resolve()

if not FINAL_RTDETR_WEIGHTS.exists():
    print(f'WARNING: RT-DETR weights not found at {FINAL_RTDETR_WEIGHTS} — skipping.')
    _rtdetr_available_b = False
else:
    _rtdetr_available_b = True

RTDETR_HP_GRID_B = {
    'conf':             [0.10, 0.15, 0.20, 0.25, 0.30, 0.40],
    'track_high_thresh': [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60],
    'track_low_thresh':  [0.01, 0.03, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40],
    'new_track_thresh':  [0.15, 0.20, 0.25, 0.35, 0.45, 0.55, 0.65],
}
RTDETR_HP_MAX_TRIALS_B = HP_MAX_TRIALS

_bt_best2     = best_configs.get("bytetrack.yaml", {})
_arch_yaml_p2 = REPO_ROOT / _arch['tracker_yaml']
import yaml as _y2b
_av2 = _y2b.safe_load(_arch_yaml_p2.read_text()) if _arch_yaml_p2.exists() else {}
_frozen_bt = {
    'track_buffer': _bt_best2.get('track_buffer', _av2.get('track_buffer', 30)),
    'match_thresh': _bt_best2.get('match_thresh', _av2.get('match_thresh', 0.60)),
}
print('Frozen ByteTrack motion knobs:')
for k, v in _frozen_bt.items(): print(f'  {k}: {v}')

def _run_tracker_rtdetr_bytetrack(weights_path, video_paths, output_dir, hp, pp_overrides=None):
    cfg = {
        'tracker_type':      'bytetrack',
        'track_high_thresh': float(hp.get('track_high_thresh', 0.25)),
        'track_low_thresh':  float(hp.get('track_low_thresh', 0.10)),
        'new_track_thresh':  float(hp.get('new_track_thresh', 0.30)),
        'track_buffer':      int(_frozen_bt['track_buffer']),
        'match_thresh':      float(_frozen_bt['match_thresh']),
        'fuse_score':        True,
    }
    tmp = Path(tempfile.mktemp(suffix='.yaml'))
    tmp.write_text(_y2b.dump(cfg))

    pp = pp_overrides or {}
    merge_time   = pp.get('pp_merge_time',  MAX_GAP      if POST_PROCESS else None)
    merge_dist   = pp.get('pp_merge_dist',  MAX_DIST_PCT if POST_PROCESS else 30.0)
    merge_interp = pp.get('pp_merge_interp', True)

    model = _RTDETR_B(str(weights_path))
    run_conf = hp.get('conf', CONF)
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    all_preds = {}
    try:
        for vp in video_paths:
            vp = Path(vp)
            if not vp.exists(): print(f'  [SKIP] {vp.name}'); continue
            results = model.track(source=str(vp), conf=run_conf, iou=IOU,
                                  tracker=str(tmp), persist=False, verbose=False, stream=True)
            rows = []
            for fi, r in enumerate(results, 1):
                if r.boxes is None or r.boxes.id is None: continue
                for box in r.boxes:
                    if box.id is None: continue
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    rows.append([fi, int(box.id.item()), x1, y1, x2-x1, y2-y1,
                                 float(box.conf.item()), int(box.cls.item())])
            rows = _merge_track_rows(rows, merge_time=merge_time,
                                     merge_dist=merge_dist, interpolate=merge_interp)
            (output_dir / f"{vp.stem}.txt").write_text("\n".join(
                f"{r[0]},{r[1]},{r[2]:.2f},{r[3]:.2f},{r[4]:.2f},{r[5]:.2f},{r[6]:.4f},{r[7]}"
                for r in rows))
            all_preds[vp.stem] = rows
    finally:
        if tmp.exists(): tmp.unlink()
    return all_preds

RTDETR_TRACKER_RUNS_B = (REPO_ROOT / 'runs/trackers_rtdetr_bytetrack').resolve()
RTDETR_TRACKER_RUNS_B.mkdir(parents=True, exist_ok=True)

rtdetr_results_b = {}
rtdetr_best_config_b = {}
rtdetr_hp_log_b = []

if _rtdetr_available_b:
    import itertools as _it3, random as _rand3
    _combos_rb = list(_it3.product(*RTDETR_HP_GRID_B.values()))
    _rand3.seed(42); _rand3.shuffle(_combos_rb)
    _combos_rb = [dict(zip(RTDETR_HP_GRID_B.keys(), c)) for c in _combos_rb[:RTDETR_HP_MAX_TRIALS_B]]
    print(f'RT-DETR/ByteTrack sweep: {len(_combos_rb)} trials')

    best_ace_rb = float('inf')
    for idx, hp in enumerate(_combos_rb):
        pred_dir_rb = RTDETR_TRACKER_RUNS_B / 'bytetrack' / 'hp_search' / f'trial_{idx:03d}'
        _run_tracker_rtdetr_bytetrack(FINAL_RTDETR_WEIGHTS, search_videos, pred_dir_rb, hp)
        m = compute_mot_metrics(MOT_DIR / 'test', pred_dir_rb)
        df_cnt = compute_count_metrics(MOT_DIR / 'test', pred_dir_rb, CLASSES)
        per_cls_r = df_cnt[df_cnt["class"] != "MACRO_AVG"]
        ace = float(per_cls_r["ACE"].mean()) if not per_cls_r.empty else None
        bias_r = float(per_cls_r["CE"].mean()) if not per_cls_r.empty else 0.0
        composite_r = (ace + BIAS_PENALTY * abs(bias_r)) if ace is not None else None
        m['ACE'] = ace
        rtdetr_hp_log_b.append({'trial': idx, **hp, **m})
        hp_str_r = " ".join(f"{k}={v}" for k, v in hp.items())
        mota_r = f"{m['MOTA']:.4f}" if m['MOTA'] is not None else "None"
        idf1_r = f"{m['IDF1']:.4f}" if m['IDF1'] is not None else "None"
        ace_r  = f"{ace:.4f}" if ace is not None else "None"
        bias_rs = f"{bias_r:.4f}"
        comp_r = f"{composite_r:.4f}" if composite_r is not None else "None"
        print(f"  [{idx+1:3d}/{len(_combos_rb)}] {hp_str_r} | MOTA={mota_r} IDF1={idf1_r} ID_Sw={m['ID_Sw']} ACE={ace_r} CE_bias={bias_rs} [composite={comp_r}]")
        if composite_r is not None and composite_r < best_ace_rb:
            best_ace_rb = ace
            rtdetr_best_config_b = hp

    pd.DataFrame(rtdetr_hp_log_b).to_csv(RTDETR_TRACKER_RUNS_B / 'bytetrack_hp_search_log.csv', index=False)
    print(f'\nBest RT-DETR ByteTrack config (ACE={best_ace_rb:.4f}):')
    print(rtdetr_best_config_b)
else:
    print('Skipped — RT-DETR weights not found.')


## Save RT-DETR/ByteTrack Final Config

In [ ]:
# ── Save RT-DETR final config (ByteTrack) ────────────────────────────────────
import json as _jsave3, yaml as _ysave3

if _rtdetr_available_b and rtdetr_best_config_b:
    _rt_hp_b    = dict(rtdetr_best_config_b)
    _rt_conf_b  = _rt_hp_b.pop('conf', CONF)

    _rt_yaml_b = REPO_ROOT / 'configs' / 'kartector_bytetrack_rtdetr_final.yaml'
    _rt_yaml_b.write_text(_ysave3.dump({
        'tracker_type':      'bytetrack',
        'track_high_thresh': _rt_hp_b.get('track_high_thresh', 0.25),
        'track_low_thresh':  _rt_hp_b.get('track_low_thresh', 0.10),
        'new_track_thresh':  _rt_hp_b.get('new_track_thresh', 0.30),
        'track_buffer':      int(_frozen_bt['track_buffer']),
        'match_thresh':      float(_frozen_bt['match_thresh']),
        'fuse_score':        True,
    }))

    _rt_cfg_b = {
        'backbone':      'rtdetr',
        'weights':       str((RTDETR_RUNS_B / 'final' / 'weights' / 'best.pt').relative_to(REPO_ROOT)),
        'tracker':       'bytetrack',
        'tracker_yaml':  str(_rt_yaml_b.relative_to(REPO_ROOT)),
        'conf':          float(_rt_conf_b),
        'iou':           float(IOU),
        'post_process':  bool(POST_PROCESS),
        'max_gap':       int(MAX_GAP),
        'max_dist_pct':  float(MAX_DIST_PCT),
        'best_hp':       rtdetr_best_config_b,
    }
    _out3 = REPO_ROOT / 'configs' / 'kartector_bytetrack_rtdetr_final.json'
    _out3.write_text(_jsave3.dumps(_rt_cfg_b, indent=2))
    print(f'RT-DETR/ByteTrack config saved: {_out3}')
    print(_jsave3.dumps(_rt_cfg_b, indent=2))
else:
    print('RT-DETR config not saved — run the sweep above first.')

## Annotated Frame Snapshots (RT-DETR)

12 evenly-spaced frames from the first eval video with RT-DETR+ByteTrack tracks overlaid.

In [ ]:
if _rtdetr_available_b and rtdetr_best_config_b and demo_video:
    from ultralytics import RTDETR as _RTDETR_VIZ

    _rt_viz_hp   = dict(rtdetr_best_config_b)
    _rt_viz_conf = _rt_viz_hp.pop('conf', CONF)
    _rt_viz_tkey = {k: v for k, v in _rt_viz_hp.items() if k in _TRACKER_KEYS}

    import tempfile as _tmpviz, yaml as _yamlviz
    _rt_viz_cfg = {
        'tracker_type':      'bytetrack',
        'track_high_thresh': _rt_viz_tkey.get('track_high_thresh', 0.25),
        'track_low_thresh':  _rt_viz_tkey.get('track_low_thresh', 0.10),
        'new_track_thresh':  _rt_viz_tkey.get('new_track_thresh', 0.30),
        'track_buffer':      int(_frozen_bt['track_buffer']),
        'match_thresh':      float(_frozen_bt['match_thresh']),
        'fuse_score':        True,
    }
    _rt_viz_tmp = Path(_tmpviz.mktemp(suffix='.yaml'))
    _rt_viz_tmp.write_text(_yamlviz.dump(_rt_viz_cfg))

    _rt_model_viz = _RTDETR_VIZ(str(FINAL_RTDETR_WEIGHTS))
    _rt_cap_viz   = cv2.VideoCapture(str(demo_video))
    _rt_total_viz = int(_rt_cap_viz.get(cv2.CAP_PROP_FRAME_COUNT)); _rt_cap_viz.release()
    _rt_results_viz = list(_rt_model_viz.track(
        source=str(demo_video), conf=_rt_viz_conf, iou=IOU,
        tracker=str(_rt_viz_tmp), persist=False, verbose=False, stream=True))
    if _rt_viz_tmp.exists(): _rt_viz_tmp.unlink()

    _rt_indices = [int(i * (_rt_total_viz - 1) / 11) for i in range(12)]
    _rt_frames_out = []
    _rt_cap2 = cv2.VideoCapture(str(demo_video))
    for _fi in _rt_indices:
        _rt_cap2.set(cv2.CAP_PROP_POS_FRAMES, _fi)
        _ret, _frame = _rt_cap2.read()
        if not _ret: continue
        _r = _rt_results_viz[min(_fi, len(_rt_results_viz) - 1)]
        _boxes, _tids, _cids, _confs = [], [], [], []
        if _r.boxes is not None and _r.boxes.id is not None:
            for _box in _r.boxes:
                if _box.id is None: continue
                _boxes.append(_box.xyxy[0].tolist())
                _tids.append(int(_box.id.item()))
                _cids.append(int(_box.cls.item()))
                _confs.append(float(_box.conf.item()))
        _rt_frames_out.append(draw_tracks(_frame, _boxes, _tids, _cids, CLASSES, _confs))
    _rt_cap2.release()

    import math as _math
    _cols = 4; _rows_n = _math.ceil(len(_rt_frames_out) / _cols)
    fig, axes = plt.subplots(_rows_n, _cols, figsize=(_cols * 4, _rows_n * 3))
    axes = axes.flatten()
    for ax, fr in zip(axes, _rt_frames_out):
        ax.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); ax.axis('off')
    for ax in axes[len(_rt_frames_out):]: ax.axis('off')
    plt.suptitle(f'RT-DETR+ByteTrack — annotated frames  ({Path(demo_video).name})', fontsize=12)
    plt.tight_layout()
    _out_snap_rt = RTDETR_TRACKER_RUNS_B / 'annotated_rtdetr_bytetrack.png'
    plt.savefig(_out_snap_rt, dpi=120, bbox_inches='tight'); plt.show()
    print(f'Saved: {_out_snap_rt}')
else:
    print('Skipped RT-DETR snapshots — weights not found or sweep not run.')

## Trajectory Plot (RT-DETR)

Centroid paths for the best RT-DETR+ByteTrack config on the demo video.

In [ ]:
if _rtdetr_available_b and rtdetr_best_config_b and demo_video:
    _rt_traj_hp   = dict(rtdetr_best_config_b)
    _rt_traj_conf = _rt_traj_hp.pop('conf', CONF)
    _rt_traj_tkey = {k: v for k, v in _rt_traj_hp.items() if k in _TRACKER_KEYS}

    import tempfile as _tmptraj2, yaml as _yamltraj2
    _rt_traj_cfg = {
        'tracker_type':      'bytetrack',
        'track_high_thresh': _rt_traj_tkey.get('track_high_thresh', 0.25),
        'track_low_thresh':  _rt_traj_tkey.get('track_low_thresh', 0.10),
        'new_track_thresh':  _rt_traj_tkey.get('new_track_thresh', 0.30),
        'track_buffer':      int(_frozen_bt['track_buffer']),
        'match_thresh':      float(_frozen_bt['match_thresh']),
        'fuse_score':        True,
    }
    _rt_traj_tmp = Path(_tmptraj2.mktemp(suffix='.yaml'))
    _rt_traj_tmp.write_text(_yamltraj2.dump(_rt_traj_cfg))

    _rt_model_traj = _RTDETR_VIZ(str(FINAL_RTDETR_WEIGHTS))
    _rt_pred_traj = {}
    for _fi2, _r2 in enumerate(_rt_model_traj.track(
            source=str(demo_video), conf=_rt_traj_conf, iou=IOU,
            tracker=str(_rt_traj_tmp), persist=False, verbose=False, stream=True), 1):
        if _r2.boxes is None or _r2.boxes.id is None: continue
        for _box2 in _r2.boxes:
            if _box2.id is None: continue
            _tid2 = int(_box2.id.item()); _cid2 = int(_box2.cls.item())
            _x1, _y1, _x2, _y2 = _box2.xyxy[0].tolist()
            _rt_pred_traj.setdefault(_tid2, []).append(((_x1+_x2)/2, (_y1+_y2)/2, _fi2, _cid2))
    if _rt_traj_tmp.exists(): _rt_traj_tmp.unlink()

    _gt_traj_rt = _load_gt_trajectories(demo_video, MOT_DIR)
    _cap_bg_rt  = cv2.VideoCapture(str(demo_video)); _ret_bg, _bg_rt = _cap_bg_rt.read(); _cap_bg_rt.release()
    if not _ret_bg: _bg_rt = np.zeros((720, 1280, 3), dtype=np.uint8)

    _ncols_rt = 2 if _gt_traj_rt else 1
    fig, axes = plt.subplots(1, _ncols_rt, figsize=(12 * _ncols_rt, 7))
    if _ncols_rt == 1: axes = [axes]
    _draw_trajectory_ax(axes[0], _rt_pred_traj,
                        f'Predicted (RT-DETR+ByteTrack)  ({Path(demo_video).name})', _bg_rt)
    if _gt_traj_rt:
        _draw_trajectory_ax(axes[1], _gt_traj_rt,
                            f'Ground Truth  ({Path(demo_video).name})', _bg_rt)
    plt.tight_layout()
    _out_traj_rt = RTDETR_TRACKER_RUNS_B / 'trajectories_rtdetr.png'
    plt.savefig(_out_traj_rt, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved: {_out_traj_rt}')
else:
    print('Skipped RT-DETR trajectory plot — weights not found or sweep not run.')
